# 빅데이터 처리와 시각화 7장 - Pandas 데이터 정제 및 처리
---

이번 장에서는 **실제 데이터 분석 업무의 80%를 차지하는 '데이터 정제'** 를 집중적으로 학습.

<div class="alert alert-block alert-info">
<b>💡 Why?</b> 현업 데이터 분석가는 전체 업무 시간의 약 80%를 데이터 수집·정제에 사용함.
분석 모델이 아무리 훌륭해도, 입력 데이터가 지저분하면 결과를 신뢰할 수 없음. (Garbage In, Garbage Out)
</div>

In [2]:
import pandas as pd
import numpy as np

---
## 1. 결측치(Missing Value) 처리 심화

#### 1-1. 결측치란 무엇인가?

- **결측치**: 데이터에 값이 없는 경우. Pandas에서는 `NaN`(Not a Number) 또는 `None`으로 표현.
- 발생 원인: 설문 미응답, 센서 오작동, 데이터 수집 오류, 시스템 이전 등 다양.

| 표현 | 설명 |
| :--- | :--- |
| `NaN` | float 기반 결측치 (numpy.nan). 숫자형 컬럼에 주로 사용 |
| `None` | 파이썬 객체형 결측치. 문자열/오브젝트 컬럼에 주로 사용 |
| `pd.NaT` | 날짜/시간형 결측치 (Not a Time) |


In [5]:
# 실습용 데이터 생성 - 결측치가 포함된 직원 정보
data_na = {
    '이름':   ['김철수', '이영희', '박민수', '최지우', '안다솜', '황준혁', '정미래', '오세진'],
    '나이':   [28,      None,    22,      31,      None,    27,      33,      26],
    '부서':   ['영업',   'IT',    '영업',   None,    'IT',    '마케팅',  None,    '마케팅'],
    '월급':   [300,     500,     None,    400,     450,     380,     None,    410],
    '평가점수': [85.5,   92.0,   78.0,   None,    88.5,   None,    75.0,   90.0]
}
df_na = pd.DataFrame(data_na)

print('=== 결측치 포함 데이터 ===')
display(df_na)

# 1. 결측치 존재 여부 확인 (True = 결측치)
print('\n--- isnull() : 결측치 위치 확인 ---')
display(df_na.isnull())

# 2. 컬럼별 결측치 수
print('\n--- 컬럼별 결측치 수 ---')
print(df_na.isnull().sum())

# 3. 결측치 비율 (%) - 실무에서 자주 사용
print('\n--- 결측치 비율 (%) ---')
print((df_na.isnull().sum() / len(df_na) * 100).round(1))

=== 결측치 포함 데이터 ===


,이름,나이,부서,월급,평가점수
0,김철수,28.0,영업,300.0,85.5
1,이영희,NaN,IT,500.0,92.0
2,박민수,22.0,영업,NaN,78.0
3,최지우,31.0,None,400.0,NaN
4,안다솜,NaN,IT,450.0,88.5
5,황준혁,27.0,마케팅,380.0,NaN
6,정미래,33.0,None,NaN,75.0
7,오세진,26.0,마케팅,410.0,90.0



--- isnull() : 결측치 위치 확인 ---


,이름,나이,부서,월급,평가점수
0,False,False,False,False,False
1,False,True,False,False,False
2,False,False,False,True,False
3,False,False,True,False,True
4,False,True,False,False,False
5,False,False,False,False,True
6,False,False,True,True,False
7,False,False,False,False,False



--- 컬럼별 결측치 수 ---
이름      0
나이      2
부서      2
월급      2
평가점수    2
dtype: int64

--- 결측치 비율 (%) ---
이름       0.0
나이      25.0
부서      25.0
월급      25.0
평가점수    25.0
dtype: float64


#### 1-2. 결측치 제거: `dropna()`

```python
df.dropna(how='any', thresh=None, subset=None)
```

| 파라미터 | 설명 | 기본값 |
| :--- | :--- | :--- |
| `how='any'` | 결측치가 **하나라도** 있는 행 제거 | 기본값 |
| `how='all'` | **모든** 값이 결측치인 행만 제거 | - |
| `thresh=n` | 결측치가 아닌 값이 **n개 미만**인 행 제거 | - |
| `subset=['컬럼']` | 특정 컬럼에서만 결측치 기준으로 제거 | - |

In [7]:
print(f'원본 행 수: {len(df_na)}\n')

# 1. how='any' : 결측치가 하나라도 있으면 제거 (기본값)
print('--- how=any (기본값): 결측치 있는 행 모두 제거 ---')
display(df_na.dropna())

# 2. how='all' : 모든 값이 결측치인 행만 제거
print('\n--- how=all: 모든 값이 결측치인 행만 제거 ---')
display(df_na.dropna(how='all'))

# 3. thresh : 유효한 값이 최소 n개 이상인 행만 유지
print('\n--- thresh=4: 유효값이 4개 이상인 행만 유지 ---')
display(df_na.dropna(thresh=4))

# 4. subset : 특정 컬럼에서만 결측치 기준 적용
print('\n--- subset=[나이, 부서]: 나이와 부서 둘 다 있는 행만 유지 ---')
display(df_na.dropna(subset=['나이', '부서']))

원본 행 수: 8

--- how=any (기본값): 결측치 있는 행 모두 제거 ---


,이름,나이,부서,월급,평가점수
0,김철수,28.0,영업,300.0,85.5
7,오세진,26.0,마케팅,410.0,90.0



--- how=all: 모든 값이 결측치인 행만 제거 ---


,이름,나이,부서,월급,평가점수
0,김철수,28.0,영업,300.0,85.5
1,이영희,NaN,IT,500.0,92.0
2,박민수,22.0,영업,NaN,78.0
3,최지우,31.0,None,400.0,NaN
4,안다솜,NaN,IT,450.0,88.5
5,황준혁,27.0,마케팅,380.0,NaN
6,정미래,33.0,None,NaN,75.0
7,오세진,26.0,마케팅,410.0,90.0



--- thresh=4: 유효값이 4개 이상인 행만 유지 ---


,이름,나이,부서,월급,평가점수
0,김철수,28.0,영업,300.0,85.5
1,이영희,NaN,IT,500.0,92.0
2,박민수,22.0,영업,NaN,78.0
4,안다솜,NaN,IT,450.0,88.5
5,황준혁,27.0,마케팅,380.0,NaN
7,오세진,26.0,마케팅,410.0,90.0



--- subset=[나이, 부서]: 나이와 부서 둘 다 있는 행만 유지 ---


,이름,나이,부서,월급,평가점수
0,김철수,28.0,영업,300.0,85.5
2,박민수,22.0,영업,NaN,78.0
5,황준혁,27.0,마케팅,380.0,NaN
7,오세진,26.0,마케팅,410.0,90.0


#### 1-3. 결측치 채우기: `fillna()`

- 결측치를 **제거하는 대신 적절한 값으로 대체**하는 방법.
- 데이터를 보존하면서 분석을 이어갈 수 있어 실무에서 더 자주 사용.

| 방법 | 코드 예시 | 설명 |
| :--- | :--- | :--- |
| 고정값 | `fillna(0)` | 결측치를 특정 값으로 대체 |
| 평균값 | `fillna(df['열'].mean())` | 숫자형 데이터에 적합 |
| 최빈값 | `fillna(df['열'].mode()[0])` | 범주형 데이터에 적합 |
| 앞값 전파 | `fillna(method='ffill')` | 시계열 데이터에 적합 |
| 뒷값 전파 | `fillna(method='bfill')` | 시계열 데이터에 적합 |

In [9]:
df_fill = df_na.copy()  # 원본 보존을 위한 복사

# 1. 숫자형 결측치: 평균값으로 채우기
age_mean = df_fill['나이'].mean()
df_fill['나이'] = df_fill['나이'].fillna(age_mean)
print(f'나이 평균: {age_mean:.1f}')

salary_median = df_fill['월급'].median()
df_fill['월급'] = df_fill['월급'].fillna(salary_median)
print(f'월급 중앙값: {salary_median}')

# 2. 범주형 결측치: 최빈값으로 채우기
dept_mode = df_fill['부서'].mode()[0]  # mode()는 Series 반환, [0]으로 첫번째 최빈값 추출
df_fill['부서'] = df_fill['부서'].fillna(dept_mode)
print(f'부서 최빈값: {dept_mode}')

# 3. 평가점수: 앞값 전파 (ffill)
df_fill['평가점수'] = df_fill['평가점수'].fillna(method='ffill')

print('\n--- fillna 적용 후 ---')
display(df_fill)
print(f'\n결측치 수:\n{df_fill.isnull().sum()}')

나이 평균: 27.8
월급 중앙값: 405.0
부서 최빈값: IT

--- fillna 적용 후 ---


/var/folders/wv/qlkf8lb51w70xm32df60g6v40000gn/T/ipykernel_33241/4077889641.py:18: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_fill['평가점수'] = df_fill['평가점수'].fillna(method='ffill')


,이름,나이,부서,월급,평가점수
0,김철수,28.000000,영업,300.0,85.5
1,이영희,27.833333,IT,500.0,92.0
2,박민수,22.000000,영업,405.0,78.0
3,최지우,31.000000,IT,400.0,78.0
4,안다솜,27.833333,IT,450.0,88.5
5,황준혁,27.000000,마케팅,380.0,88.5
6,정미래,33.000000,IT,405.0,75.0
7,오세진,26.000000,마케팅,410.0,90.0



결측치 수:
이름      0
나이      0
부서      0
월급      0
평가점수    0
dtype: int64


#### 1-4. 보간법: `interpolate()`

- **보간법(Interpolation)**: 앞뒤 값을 기반으로 결측치를 **수학적으로 추정**하는 방법.
- 특히 **시계열 데이터**(주가, 온도, 센서 데이터 등)에서 매우 유용.

<div class="alert alert-block alert-info">
<b>💡 예시</b> 1월: 100, 2월: NaN, 3월: 150 → 보간 후 2월: 125 (선형 보간)
</div>

In [11]:
# 월별 매출 데이터 (일부 누락)
monthly_sales = pd.DataFrame({
    '월': ['1월', '2월', '3월', '4월', '5월', '6월', '7월', '8월'],
    '매출': [100, None, 130, None, None, 160, None, 200]
})

print('=== 원본 데이터 ===')
display(monthly_sales)

# 1. 선형 보간 (linear): 앞뒤 값의 직선 보간
monthly_sales['선형보간'] = monthly_sales['매출'].interpolate(method='linear')

# 2. ffill과 비교
monthly_sales['앞값전파'] = monthly_sales['매출'].fillna(method='ffill')

print('\n=== 보간법 비교 ===')
display(monthly_sales)

print('\n💡 선형보간: 앞뒤 값 사이를 균등하게 나눠 채움')
print('   앞값전파: 가장 최근 유효값으로 채움')

=== 원본 데이터 ===


,월,매출
0,1월,100.0
1,2월,NaN
2,3월,130.0
3,4월,NaN
4,5월,NaN
5,6월,160.0
6,7월,NaN
7,8월,200.0



=== 보간법 비교 ===


/var/folders/wv/qlkf8lb51w70xm32df60g6v40000gn/T/ipykernel_33241/21042698.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  monthly_sales['앞값전파'] = monthly_sales['매출'].fillna(method='ffill')


,월,매출,선형보간,앞값전파
0,1월,100.0,100.0,100.0
1,2월,NaN,115.0,100.0
2,3월,130.0,130.0,130.0
3,4월,NaN,140.0,130.0
4,5월,NaN,150.0,130.0
5,6월,160.0,160.0,160.0
6,7월,NaN,180.0,160.0
7,8월,200.0,200.0,200.0



💡 선형보간: 앞뒤 값 사이를 균등하게 나눠 채움
   앞값전파: 가장 최근 유효값으로 채움


> #### Exercise
> 아래 `df_ex1` 데이터를 사용하여 다음을 수행하시오.
> 1. 각 컬럼별 결측치 수와 결측치 비율(%)을 출력하시오.
> 2. `점수` 컬럼의 결측치를 **중앙값**으로 채우시오.
> 3. `도시` 컬럼의 결측치를 **최빈값**으로 채우시오.
> 4. `방문횟수` 컬럼의 결측치가 있는 행을 **제거**하시오. (단, 나머지 컬럼의 결측치는 이미 채워진 상태 기준)


In [13]:
df_ex1 = pd.DataFrame({
    '고객명': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'],
    '도시': ['서울', '부산', None, '서울', '대구', None, '서울', '부산'],
    '점수': [85, None, 72, 90, None, 68, 88, None],
    '방문횟수': [5, 3, None, 8, 2, None, 4, 6]
})
display(df_ex1)

# 여기에 코드를 작성하시오.

,고객명,도시,점수,방문횟수
0,A,서울,85.0,5.0
1,B,부산,NaN,3.0
2,C,None,72.0,NaN
3,D,서울,90.0,8.0
4,E,대구,NaN,2.0
5,F,None,68.0,NaN
6,G,서울,88.0,4.0
7,H,부산,NaN,6.0


> #### Advance
> - 아래 `df_adv1` 은 일별 기온 데이터.
> - 결측치 비율이 **30% 이상인 컬럼은 제거**하고, 나머지 컬럼의 결측치는 **선형 보간법**으로 채우시오.
> - 최종 데이터에 결측치가 없는지 확인하시오.

In [15]:
df_adv1 = pd.DataFrame({
    '날짜': pd.date_range('2024-01-01', periods=10),
    '서울': [2.1, None, 3.5, 4.0, None, 5.2, 5.8, None, 6.3, 7.0],
    '부산': [8.5, 9.0, None, 10.1, 10.5, None, 11.2, 11.8, None, 12.5],
    '제주': [None, None, None, 15.0, None, None, None, 17.0, None, None]
})
display(df_adv1)

# 여기에 코드를 작성하시오.


,날짜,서울,부산,제주
0,2024-01-01,2.1,8.5,NaN
1,2024-01-02,NaN,9.0,NaN
2,2024-01-03,3.5,NaN,NaN
3,2024-01-04,4.0,10.1,15.0
4,2024-01-05,NaN,10.5,NaN
5,2024-01-06,5.2,NaN,NaN
6,2024-01-07,5.8,11.2,NaN
7,2024-01-08,NaN,11.8,17.0
8,2024-01-09,6.3,NaN,NaN
9,2024-01-10,7.0,12.5,NaN


---
## 2. 중복 데이터 처리
- 동일한 데이터가 여러 번 입력되면 통계 왜곡, 분석 오류가 발생함.
- **중복 여부를 먼저 확인하고, 적절한 기준으로 제거**하는 것이 핵심임.

#### 2-1. 중복 데이터 확인: `duplicated()`

```python
df.duplicated(subset=None, keep='first')
```

| `keep` 옵션 | 설명 |
| :--- | :--- |
| `'first'` | 처음 등장한 행은 False(유지), 이후 중복은 True(중복) |
| `'last'` | 마지막 등장한 행은 False(유지), 이전 중복은 True(중복) |
| `False` | 중복된 모든 행을 True로 표시 |

In [18]:
# 중복 데이터가 포함된 데이터프레임
data_dup = {
    '직원ID': [101, 102, 103, 102, 104, 101],
    '이름':   ['김철수', '이영희', '박민수', '이영희', '최지우', '김철수'],
    '부서':   ['영업',   'IT',    '영업',   'IT',    '인사',   '영업'],
    '월급':   [300,     500,     250,     500,     400,     300]
}
df_dup = pd.DataFrame(data_dup)
display(df_dup)

print('--- duplicated() : 중복 여부 확인 (True = 중복) ---')
print(df_dup.duplicated())

print('\n--- 중복 행만 출력 ---')
display(df_dup[df_dup.duplicated()])

print('\n--- keep=False : 중복에 연루된 모든 행 표시 ---')
display(df_dup[df_dup.duplicated(keep=False)])

,직원ID,이름,부서,월급
0,101,김철수,영업,300
1,102,이영희,IT,500
2,103,박민수,영업,250
3,102,이영희,IT,500
4,104,최지우,인사,400
5,101,김철수,영업,300


--- duplicated() : 중복 여부 확인 (True = 중복) ---
0    False
1    False
2    False
3     True
4    False
5     True
dtype: bool

--- 중복 행만 출력 ---


,직원ID,이름,부서,월급
3,102,이영희,IT,500
5,101,김철수,영업,300



--- keep=False : 중복에 연루된 모든 행 표시 ---


,직원ID,이름,부서,월급
0,101,김철수,영업,300
1,102,이영희,IT,500
3,102,이영희,IT,500
5,101,김철수,영업,300


#### 2-2. 중복 제거: `drop_duplicates()`

- `subset`: 특정 컬럼 기준으로만 중복 판단 (전체 행이 아닌 일부 컬럼 기준)


In [20]:
# 1. 모든 컬럼 기준으로 중복 제거 (기본값)
print('--- 전체 컬럼 기준 중복 제거 ---')
display(df_dup.drop_duplicates())

# 2. 특정 컬럼(직원ID) 기준으로만 중복 제거
print('\n--- 직원ID 기준 중복 제거 (처음 등장한 행 유지) ---')
display(df_dup.drop_duplicates(subset=['직원ID'], keep='first'))

# 3. keep='last': 마지막 등장한 행 유지
print('\n--- 직원ID 기준 중복 제거 (마지막 등장한 행 유지) ---')
display(df_dup.drop_duplicates(subset=['직원ID'], keep='last'))

print(f'\n원본: {len(df_dup)}행 → 제거 후: {len(df_dup.drop_duplicates(subset=["직원ID"]))}행')

--- 전체 컬럼 기준 중복 제거 ---


,직원ID,이름,부서,월급
0,101,김철수,영업,300
1,102,이영희,IT,500
2,103,박민수,영업,250
4,104,최지우,인사,400



--- 직원ID 기준 중복 제거 (처음 등장한 행 유지) ---


,직원ID,이름,부서,월급
0,101,김철수,영업,300
1,102,이영희,IT,500
2,103,박민수,영업,250
4,104,최지우,인사,400



--- 직원ID 기준 중복 제거 (마지막 등장한 행 유지) ---


,직원ID,이름,부서,월급
2,103,박민수,영업,250
3,102,이영희,IT,500
4,104,최지우,인사,400
5,101,김철수,영업,300



원본: 6행 → 제거 후: 4행


> #### Exercise
> 아래 `df_ex2` 데이터에서 다음을 수행하시오.
> 1. 전체 컬럼 기준으로 완전히 동일한 중복 행의 수를 출력하시오.
> 2. `주문ID` 컬럼 기준으로 중복된 행을 찾아서 출력하시오.
> 3. `주문ID` 기준 중복 제거 후, 가장 최신 주문(마지막 행)만 남기시오.


In [22]:
df_ex2 = pd.DataFrame({
    '주문ID': ['A001', 'A002', 'A001', 'A003', 'A002', 'A004'],
    '고객':   ['김철수', '이영희', '김철수', '박민수', '이영희', '최지우'],
    '상품':   ['노트북', '마우스', '노트북', '모니터', '키보드', '웹캠'],
    '금액':   [1200000, 35000, 1200000, 250000, 120000, 85000]
})
display(df_ex2)

# 여기에 코드를 작성하시오.


,주문ID,고객,상품,금액
0,A001,김철수,노트북,1200000
1,A002,이영희,마우스,35000
2,A001,김철수,노트북,1200000
3,A003,박민수,모니터,250000
4,A002,이영희,키보드,120000
5,A004,최지우,웹캠,85000


---
## 3. 데이터 타입 변환
- 실제 데이터에서는 **숫자가 문자열로**, **날짜가 일반 문자열로** 저장되는 경우가 매우 흔함.
- 올바른 타입 변환 없이는 연산, 정렬, 날짜 계산이 불가능함.

#### 3-1. 타입 확인과 기본 변환: `astype()`

- `df.dtypes`: 각 컬럼의 데이터 타입 확인
- `df['컬럼'].astype(타입)`: 타입 변환 (변환 불가한 값이 있으면 에러 발생)

In [25]:
# 타입 변환이 필요한 상품 데이터
data_type = {
    '상품명':  ['노트북',     '마우스',   '모니터',    '키보드',    '웹캠'],
    '가격':   ['1200000',   '35000',   '250000',   '120000',   '85000'],  # 문자열!
    '수량':   ['10',        '50',      '5',        '20',       '15'],     # 문자열!
    '출시일':  ['2024-01-15', '2023-08-20', '2024-03-10', '2023-11-05', '2024-02-28'],
    '카테고리': ['전자',      '주변기기',  '전자',      '주변기기',  '주변기기']
}
df_type = pd.DataFrame(data_type)

print('=== 변환 전 데이터 타입 ===')
print(df_type.dtypes)

# astype() 으로 타입 변환
df_type['가격'] = df_type['가격'].astype(int)
df_type['수량'] = df_type['수량'].astype(int)

print('\n=== 변환 후 데이터 타입 ===')
print(df_type.dtypes)

# 이제 연산 가능!
df_type['총액'] = df_type['가격'] * df_type['수량']
display(df_type)

=== 변환 전 데이터 타입 ===
상품명     object
가격      object
수량      object
출시일     object
카테고리    object
dtype: object

=== 변환 후 데이터 타입 ===
상품명     object
가격       int64
수량       int64
출시일     object
카테고리    object
dtype: object


,상품명,가격,수량,출시일,카테고리,총액
0,노트북,1200000,10,2024-01-15,전자,12000000
1,마우스,35000,50,2023-08-20,주변기기,1750000
2,모니터,250000,5,2024-03-10,전자,1250000
3,키보드,120000,20,2023-11-05,주변기기,2400000
4,웹캠,85000,15,2024-02-28,주변기기,1275000


#### 3-2. 오류 허용 변환: `pd.to_numeric()`

- `astype(int)`는 변환 불가 값(예: `'N/A'`, `'abc'`)이 있으면 에러 발생.
- `pd.to_numeric(errors='coerce')`: 변환 불가 값을 **NaN으로 변환** (에러 없이 처리 가능).

In [27]:
# 변환 불가 값이 섞인 월급 데이터
salary_series = pd.Series(['300', '500', '250', 'N/A', '450', '오류', '410'])
print('원본:', salary_series.tolist())

# astype 시도 → 에러 발생
try:
    salary_series.astype(int)
except ValueError as e:
    print(f'\nastype 에러: {e}')

# pd.to_numeric(errors='coerce') → NaN으로 안전하게 변환
salary_numeric = pd.to_numeric(salary_series, errors='coerce')
print('\npd.to_numeric 결과:')
print(salary_numeric)

# NaN을 중앙값으로 채우기
salary_clean = salary_numeric.fillna(salary_numeric.median())
print('\nNaN 채운 후:')
print(salary_clean)

원본: ['300', '500', '250', 'N/A', '450', '오류', '410']

astype 에러: invalid literal for int() with base 10: 'N/A'

pd.to_numeric 결과:
0    300.0
1    500.0
2    250.0
3      NaN
4    450.0
5      NaN
6    410.0
dtype: float64

NaN 채운 후:
0    300.0
1    500.0
2    250.0
3    410.0
4    450.0
5    410.0
6    410.0
dtype: float64


#### 3-3. category 타입

- 반복되는 문자열 값(성별, 지역, 등급 등)을 **category 타입**으로 변환하면:
  - 메모리 사용량이 **크게 감소** (반복 문자열을 숫자 코드로 저장)
  - **정렬 순서** 지정 가능 (예: 사원 < 대리 < 과장 < 차장 < 부장)

<div class="alert alert-block alert-info">
<b>💡 실무 팁</b> 수백만 행 데이터에서 '성별', '지역', '등급' 같은 반복 컬럼을 category로 바꾸면 메모리를 절약할 수 있음.
</div>

In [29]:
# 직급 데이터 (반복 값이 많은 범주형)
ranks = pd.Series(['사원', '대리', '과장', '사원', '부장', '대리', '차장', '사원', '과장', '대리'])

# 1. 메모리 비교
print(f'object 타입 메모리: {ranks.memory_usage(deep=True)} bytes')

ranks_cat = ranks.astype('category')
print(f'category 타입 메모리: {ranks_cat.memory_usage(deep=True)} bytes')

# 2. 순서 있는 category (Ordered Category) - 직급 순서 지정
rank_order = ['사원', '대리', '과장', '차장', '부장']
ranks_ordered = pd.Categorical(ranks, categories=rank_order, ordered=True)

print('\n--- 순서 있는 category ---')
print(ranks_ordered)

# 3. 순서 기반 비교 가능!
print('\n과장보다 직급이 높은가?', ranks_ordered > '과장')

# 4. DataFrame에서 활용
df_type['카테고리'] = df_type['카테고리'].astype('category')
print('\n카테고리 목록:', df_type['카테고리'].cat.categories.tolist())

object 타입 메모리: 832 bytes
category 타입 메모리: 699 bytes

--- 순서 있는 category ---
['사원', '대리', '과장', '사원', '부장', '대리', '차장', '사원', '과장', '대리']
Categories (5, object): ['사원' < '대리' < '과장' < '차장' < '부장']

과장보다 직급이 높은가? [False False False False  True False  True False False False]

카테고리 목록: ['전자', '주변기기']


> #### Exercise
> 아래 `df_ex3` 데이터를 보고 다음을 수행하시오.
> 1. 각 컬럼의 데이터 타입을 확인하시오.
> 2. `판매량` 컬럼을 정수형(int)으로 변환하시오. (단, 변환 불가한 값은 NaN으로 처리 후 0으로 채울 것)
> 3. `단가` 컬럼을 실수형(float)으로 변환하시오.
> 4. `판매량 × 단가`로 `매출` 컬럼을 새로 추가하시오.

In [31]:
df_ex3 = pd.DataFrame({
    '상품': ['A', 'B', 'C', 'D', 'E'],
    '판매량': ['100', '200', '미입력', '150', '300'],
    '단가': ['5000', '3000', '8000', '4500', '2000']
})
display(df_ex3)

# 여기에 코드를 작성하시오.


,상품,판매량,단가
0,A,100,5000
1,B,200,3000
2,C,미입력,8000
3,D,150,4500
4,E,300,2000


---
## 4. 문자열 데이터 처리 (str accessor)
- 실무 데이터에는 공백, 대소문자 불일치, 불필요한 특수문자 등이 자주 포함됨.
- Pandas의 **str accessor**를 사용하면 Python 문자열 함수를 DataFrame 전체에 한 번에 적용할 수 있음.

#### 4-1. str accessor란?

- `df['컬럼'].str.함수()` 형태로 사용
- 파이썬의 문자열 함수를 Series 전체에 벡터화하여 적용

| 함수 | 설명 | 예시 |
| :--- | :--- | :--- |
| `str.strip()` | 앞뒤 공백 제거 | `'  김철수 '` → `'김철수'` |
| `str.upper()` | 대문자 변환 | `'hello'` → `'HELLO'` |
| `str.lower()` | 소문자 변환 | `'HELLO'` → `'hello'` |
| `str.replace(a, b)` | 문자열 치환 | `'010-1234'` → `'0101234'` |
| `str.contains(s)` | 특정 문자 포함 여부 (불리언 반환) | `'서울시 강남구'` → `True` |
| `str.split(s)` | 문자열 분리 | `'홍길동_영업팀'` → `['홍길동', '영업팀']` |
| `str.len()` | 문자열 길이 | `'hello'` → `5` |

In [34]:
# 문자열 처리가 필요한 데이터
data_str = {
    '이름':   ['  김철수 ',  '이영희',   '박 민수',   '최지우  ',  '안다솜'],
    '이메일':  ['Kim@Company.com', 'lee@company.com', 'PARK@Company.com', 'choi@COMPANY.com', 'ahn@company.com'],
    '전화번호': ['010-1234-5678', '01056789012', '010.9012.3456', '010-3456-7890', '010 7890 1234'],
    '직급팀':  ['대리_영업팀', '과장_IT팀', '차장_마케팅팀', '부장_인사팀', '사원_영업팀']
}
df_str = pd.DataFrame(data_str)

print('=== 원본 데이터 ===')
display(df_str)

# 1. 공백 제거
df_str['이름'] = df_str['이름'].str.strip()
print('\n공백 제거 후 이름:', df_str['이름'].tolist())

# 2. 소문자 통일 (이메일 정규화)
df_str['이메일'] = df_str['이메일'].str.lower()
print('소문자 변환 후 이메일:', df_str['이메일'].tolist())

# 3. 전화번호 정규화 (구분자 제거 후 표준 형식으로)
df_str['전화번호'] = df_str['전화번호'].str.replace('-', '').str.replace('.', '').str.replace(' ', '')
print('구분자 제거 후 전화번호:', df_str['전화번호'].tolist())

=== 원본 데이터 ===


,이름,이메일,전화번호,직급팀
0,김철수,Kim@Company.com,010-1234-5678,대리_영업팀
1,이영희,lee@company.com,01056789012,과장_IT팀
2,박 민수,PARK@Company.com,010.9012.3456,차장_마케팅팀
3,최지우,choi@COMPANY.com,010-3456-7890,부장_인사팀
4,안다솜,ahn@company.com,010 7890 1234,사원_영업팀



공백 제거 후 이름: ['김철수', '이영희', '박 민수', '최지우', '안다솜']
소문자 변환 후 이메일: ['kim@company.com', 'lee@company.com', 'park@company.com', 'choi@company.com', 'ahn@company.com']
구분자 제거 후 전화번호: ['01012345678', '01056789012', '01090123456', '01034567890', '01078901234']


#### 4-2. 문자열 필터링: `str.contains()`

- 특정 문자열을 포함하는 행만 추출할 때 사용
- `na=False`: 결측치를 False로 처리 (에러 방지)

In [36]:
# 문자열 필터링
print('--- 이메일에 company가 포함된 직원 ---')
display(df_str[df_str['이메일'].str.contains('company', na=False)])

# startswith / endswith
print('\n--- 이메일이 kim으로 시작하는 직원 ---')
display(df_str[df_str['이메일'].str.startswith('kim')])

print('\n--- 이메일이 .com으로 끝나는 직원 ---')
display(df_str[df_str['이메일'].str.endswith('.com')])

# 문자열 길이
print('\n--- 직급팀 문자열 길이 ---')
df_str['직급팀_길이'] = df_str['직급팀'].str.len()
display(df_str[['직급팀', '직급팀_길이']])

--- 이메일에 company가 포함된 직원 ---


,이름,이메일,전화번호,직급팀
0,김철수,kim@company.com,01012345678,대리_영업팀
1,이영희,lee@company.com,01056789012,과장_IT팀
2,박 민수,park@company.com,01090123456,차장_마케팅팀
3,최지우,choi@company.com,01034567890,부장_인사팀
4,안다솜,ahn@company.com,01078901234,사원_영업팀



--- 이메일이 kim으로 시작하는 직원 ---


,이름,이메일,전화번호,직급팀
0,김철수,kim@company.com,01012345678,대리_영업팀



--- 이메일이 .com으로 끝나는 직원 ---


,이름,이메일,전화번호,직급팀
0,김철수,kim@company.com,01012345678,대리_영업팀
1,이영희,lee@company.com,01056789012,과장_IT팀
2,박 민수,park@company.com,01090123456,차장_마케팅팀
3,최지우,choi@company.com,01034567890,부장_인사팀
4,안다솜,ahn@company.com,01078901234,사원_영업팀



--- 직급팀 문자열 길이 ---


,직급팀,직급팀_길이
0,대리_영업팀,6
1,과장_IT팀,6
2,차장_마케팅팀,7
3,부장_인사팀,6
4,사원_영업팀,6


#### 4-3. 문자열 분리: `str.split()`

- `str.split(구분자, expand=True)`: 분리 결과를 **별도 컬럼으로 확장**
- `str.split(구분자).str[n]`: 분리 결과에서 **n번째 요소만** 추출

In [38]:
# '직급팀' 컬럼에서 직급과 팀 분리
print('=== 직급팀 분리 전 ===')
print(df_str['직급팀'].tolist())

# expand=True: 분리된 각 부분을 별도 컬럼으로
split_result = df_str['직급팀'].str.split('_', expand=True)
split_result.columns = ['직급', '팀']
print('\n--- expand=True 결과 ---')
display(split_result)

# 원래 DataFrame에 추가
df_str['직급'] = df_str['직급팀'].str.split('_').str[0]
df_str['팀']   = df_str['직급팀'].str.split('_').str[1]

print('\n=== 최종 정리된 데이터 ===')
display(df_str[['이름', '직급', '팀', '이메일', '전화번호']])

=== 직급팀 분리 전 ===
['대리_영업팀', '과장_IT팀', '차장_마케팅팀', '부장_인사팀', '사원_영업팀']

--- expand=True 결과 ---


,직급,팀
0,대리,영업팀
1,과장,IT팀
2,차장,마케팅팀
3,부장,인사팀
4,사원,영업팀



=== 최종 정리된 데이터 ===


,이름,직급,팀,이메일,전화번호
0,김철수,대리,영업팀,kim@company.com,01012345678
1,이영희,과장,IT팀,lee@company.com,01056789012
2,박 민수,차장,마케팅팀,park@company.com,01090123456
3,최지우,부장,인사팀,choi@company.com,01034567890
4,안다솜,사원,영업팀,ahn@company.com,01078901234


> #### Exercise
> 아래 `df_ex4` 데이터를 사용하여 다음을 수행하시오.
> 1. `주소` 컬럼에서 '서울'이 포함된 고객만 필터링하시오.
> 2. `상품코드` 컬럼에서 `'-'`를 기준으로 분리하여 `카테고리`(앞부분)와 `번호`(뒷부분) 컬럼을 만드시오.
> 3. `고객명` 컬럼의 앞뒤 공백을 제거하시오.

```python
df_ex4 = pd.DataFrame({
    '고객명': ['  김철수', '이영희  ', ' 박민수 ', '최지우', '  안다솜'],
    '주소': ['서울시 강남구', '부산시 해운대구', '서울시 마포구', '대구시 중구', '서울시 종로구'],
    '상품코드': ['ELEC-001', 'ACC-002', 'ELEC-003', 'ACC-004', 'ELEC-005']
})
```

In [40]:
df_ex4 = pd.DataFrame({
    '고객명': ['  김철수', '이영희  ', ' 박민수 ', '최지우', '  안다솜'],
    '주소': ['서울시 강남구', '부산시 해운대구', '서울시 마포구', '대구시 중구', '서울시 종로구'],
    '상품코드': ['ELEC-001', 'ACC-002', 'ELEC-003', 'ACC-004', 'ELEC-005']
})
display(df_ex4)

# 여기에 코드를 작성하시오.


,고객명,주소,상품코드
0,김철수,서울시 강남구,ELEC-001
1,이영희,부산시 해운대구,ACC-002
2,박민수,서울시 마포구,ELEC-003
3,최지우,대구시 중구,ACC-004
4,안다솜,서울시 종로구,ELEC-005


> #### Advance
> - 아래 `df_adv4`의 `이메일` 컬럼에서 `@` 앞부분(아이디)과 뒷부분(도메인)을 각각 추출하여 별도 컬럼으로 만드시오.
> - `도메인` 컬럼 기준으로 회사 이메일(company.com)과 개인 이메일(gmail.com, naver.com)을 구분하는 `이메일_유형` 컬럼을 추가하시오.

```python
df_adv4 = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수', '최지우', '안다솜'],
    '이메일': ['kim@company.com', 'lee@gmail.com', 'park@naver.com', 'choi@company.com', 'ahn@gmail.com']
})
```

In [42]:
df_adv4 = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수', '최지우', '안다솜'],
    '이메일': ['kim@company.com', 'lee@gmail.com', 'park@naver.com', 'choi@company.com', 'ahn@gmail.com']
})
display(df_adv4)

# 여기에 코드를 작성하시오.


,이름,이메일
0,김철수,kim@company.com
1,이영희,lee@gmail.com
2,박민수,park@naver.com
3,최지우,choi@company.com
4,안다솜,ahn@gmail.com


---
## 5. 날짜/시간 데이터 처리 (dt accessor)
- 날짜 데이터는 문자열로 저장되는 경우가 많아 **`pd.to_datetime()`으로 변환**이 필요함.
- 변환 후 **dt accessor**를 사용하면 연, 월, 일, 요일 등 다양한 정보를 쉽게 추출할 수 있음.

#### 5-1. 날짜 타입 변환: `pd.to_datetime()`

- 다양한 형식의 날짜 문자열을 `datetime64` 타입으로 변환
- `format` 파라미터로 형식 명시 가능: `%Y`(년), `%m`(월), `%d`(일)

In [45]:
# 날짜 처리 실습 데이터
data_date = {
    '직원명':      ['김철수',       '이영희',       '박민수',       '최지우',       '안다솜'],
    '입사일':      ['2022-01-01',   '2021-03-15',   '2023-05-10',   '2020-11-20',   '2022-07-01'],
    '생년월일':     ['1994-05-20',   '1989-11-08',   '2001-03-25',   '1993-07-15',   '1995-12-30'],
    '마지막_교육일': ['2025-01-10',   '2024-11-20',   '2025-03-05',   '2024-09-15',   '2025-02-28']
}
df_date = pd.DataFrame(data_date)

print('변환 전 타입:')
print(df_date.dtypes)

# pd.to_datetime() 으로 변환
df_date['입사일']      = pd.to_datetime(df_date['입사일'])
df_date['생년월일']    = pd.to_datetime(df_date['생년월일'])
df_date['마지막_교육일'] = pd.to_datetime(df_date['마지막_교육일'])

print('\n변환 후 타입:')
print(df_date.dtypes)
display(df_date)

변환 전 타입:
직원명        object
입사일        object
생년월일       object
마지막_교육일    object
dtype: object

변환 후 타입:
직원명                object
입사일        datetime64[ns]
생년월일       datetime64[ns]
마지막_교육일    datetime64[ns]
dtype: object


,직원명,입사일,생년월일,마지막_교육일
0,김철수,2022-01-01,1994-05-20,2025-01-10
1,이영희,2021-03-15,1989-11-08,2024-11-20
2,박민수,2023-05-10,2001-03-25,2025-03-05
3,최지우,2020-11-20,1993-07-15,2024-09-15
4,안다솜,2022-07-01,1995-12-30,2025-02-28


#### 5-2. dt accessor - 날짜 요소 추출

| dt 속성 | 설명 | 반환 예시 |
| :--- | :--- | :--- |
| `dt.year` | 연도 | 2024 |
| `dt.month` | 월 | 3 |
| `dt.day` | 일 | 15 |
| `dt.dayofweek` | 요일 번호 (0=월요일, 6=일요일) | 4 |
| `dt.day_name()` | 요일 이름 (영어) | 'Friday' |
| `dt.quarter` | 분기 (1~4) | 1 |
| `dt.strftime(fmt)` | 원하는 형식의 문자열로 변환 | '2024년 03월 15일' |

In [47]:
# dt accessor 로 날짜 요소 추출
df_date['입사_연도']   = df_date['입사일'].dt.year
df_date['입사_월']    = df_date['입사일'].dt.month
df_date['입사_분기']   = df_date['입사일'].dt.quarter
df_date['입사_요일번호'] = df_date['입사일'].dt.dayofweek  # 0=월, 6=일
df_date['입사_요일']   = df_date['입사일'].dt.day_name()

print('=== dt accessor 활용 ===')
display(df_date[['직원명', '입사일', '입사_연도', '입사_월', '입사_분기', '입사_요일번호', '입사_요일']])

# 특정 형식의 문자열로 변환
df_date['입사일_표시'] = df_date['입사일'].dt.strftime('%Y년 %m월 %d일')
print('\n--- 날짜 형식 변환 ---')
print(df_date['입사일_표시'].tolist())

=== dt accessor 활용 ===


,직원명,입사일,입사_연도,입사_월,입사_분기,입사_요일번호,입사_요일
0,김철수,2022-01-01,2022,1,1,5,Saturday
1,이영희,2021-03-15,2021,3,1,0,Monday
2,박민수,2023-05-10,2023,5,2,2,Wednesday
3,최지우,2020-11-20,2020,11,4,4,Friday
4,안다솜,2022-07-01,2022,7,3,4,Friday



--- 날짜 형식 변환 ---
['2022년 01월 01일', '2021년 03월 15일', '2023년 05월 10일', '2020년 11월 20일', '2022년 07월 01일']


#### 5-3. 날짜 연산

- datetime 타입끼리 **빼면** `timedelta` (기간)를 반환
- `pd.Timestamp.now()`: 현재 날짜/시간
- `pd.DateOffset`: 날짜에 기간 더하기/빼기

In [49]:
today = pd.Timestamp('2026-05-06')  # 오늘 날짜 기준

# 1. 근속 연수 계산 (일수 → 년수)
df_date['근속일수'] = (today - df_date['입사일']).dt.days
df_date['근속년수'] = (df_date['근속일수'] / 365).round(1)

# 2. 나이 계산
df_date['나이'] = today.year - df_date['생년월일'].dt.year

# 3. 마지막 교육일로부터 경과일
df_date['교육_경과일'] = (today - df_date['마지막_교육일']).dt.days

print('=== 날짜 연산 결과 ===')
display(df_date[['직원명', '입사일', '근속일수', '근속년수', '나이', '교육_경과일']])

# 4. 교육 필요자 필터링 (마지막 교육 후 180일 이상 경과)
print('\n--- 교육 필요 직원 (180일 이상 경과) ---')
display(df_date[df_date['교육_경과일'] >= 180][['직원명', '마지막_교육일', '교육_경과일']])

=== 날짜 연산 결과 ===


,직원명,입사일,근속일수,근속년수,나이,교육_경과일
0,김철수,2022-01-01,1586,4.3,32,481
1,이영희,2021-03-15,1878,5.1,37,532
2,박민수,2023-05-10,1092,3.0,25,427
3,최지우,2020-11-20,1993,5.5,33,598
4,안다솜,2022-07-01,1405,3.8,31,432



--- 교육 필요 직원 (180일 이상 경과) ---


,직원명,마지막_교육일,교육_경과일
0,김철수,2025-01-10,481
1,이영희,2024-11-20,532
2,박민수,2025-03-05,427
3,최지우,2024-09-15,598
4,안다솜,2025-02-28,432


> #### Exercise
> 아래 `df_ex5` 주문 데이터를 사용하여 다음을 수행하시오.
> 1. `주문일` 컬럼을 datetime 타입으로 변환하시오.
> 2. `주문_년도`, `주문_월`, `주문_요일이름` 컬럼을 추가하시오.
> 3. 2024년 2분기(4~6월)에 접수된 주문만 필터링하시오.
> 4. 주문일로부터 배송까지 `리드타임(일수)`을 계산하여 추가하시오. (기준일: 2026-05-06)

In [51]:
df_ex5 = pd.DataFrame({
    '주문ID': ['O001', 'O002', 'O003', 'O004', 'O005', 'O006'],
    '주문일': ['2024-01-15', '2024-04-20', '2024-06-30', '2024-09-05', '2024-11-11', '2025-02-14'],
    '상품': ['노트북', '마우스', '모니터', '키보드', '웹캠', '노트북'],
    '금액': [1200000, 35000, 250000, 120000, 85000, 1200000]
})
display(df_ex5)

# 여기에 코드를 작성하시오.


,주문ID,주문일,상품,금액
0,O001,2024-01-15,노트북,1200000
1,O002,2024-04-20,마우스,35000
2,O003,2024-06-30,모니터,250000
3,O004,2024-09-05,키보드,120000
4,O005,2024-11-11,웹캠,85000
5,O006,2025-02-14,노트북,1200000


---
## 6. 데이터 그룹화 (GroupBy)
- **GroupBy**는 '특정 기준으로 데이터를 묶고 → 각 그룹에 집계 함수를 적용'하는 패턴임.
- SQL의 `GROUP BY`와 동일한 개념으로, 부서별 평균 월급, 카테고리별 매출 합계 등을 계산할 때 필수.

#### 6-1. GroupBy 기본 사용법

```python
df.groupby('기준컬럼')['집계컬럼'].집계함수()
df.groupby(['기준1', '기준2'])['집계컬럼'].집계함수()
```

In [54]:
# GroupBy 실습 데이터
data_gb = {
    '이름':    ['김철수', '이영희', '박민수', '최지우', '안다솜', '황준혁', '정미래', '오세진', '류하늘', '홍길동'],
    '부서':    ['영업',   'IT',    '영업',   '인사',   'IT',    '마케팅', '영업',   'IT',    '마케팅', '인사'],
    '직급':    ['대리',   '과장',  '사원',   '부장',   '대리',   '과장',  '과장',   '사원',  '대리',   '차장'],
    '월급':    [300,     500,     250,     600,     450,     480,     420,     280,     390,     550],
    '판매실적': [850,     None,    920,     None,    760,     640,     980,     None,    720,     None]
}
df_gb = pd.DataFrame(data_gb)
display(df_gb)

# 1. 부서별 평균 월급
print('\n--- 부서별 평균 월급 ---')
print(df_gb.groupby('부서')['월급'].mean())

# 2. 부서별 직원 수 (count)
print('\n--- 부서별 직원 수 ---')
print(df_gb.groupby('부서')['이름'].count())

# 3. 직급별 월급 최대/최소
print('\n--- 직급별 월급 최대/최소 ---')
print(df_gb.groupby('직급')['월급'].agg(['max', 'min']))

,이름,부서,직급,월급,판매실적
0,김철수,영업,대리,300,850.0
1,이영희,IT,과장,500,NaN
2,박민수,영업,사원,250,920.0
3,최지우,인사,부장,600,NaN
4,안다솜,IT,대리,450,760.0
5,황준혁,마케팅,과장,480,640.0
6,정미래,영업,과장,420,980.0
7,오세진,IT,사원,280,NaN
8,류하늘,마케팅,대리,390,720.0
9,홍길동,인사,차장,550,NaN



--- 부서별 평균 월급 ---
부서
IT     410.000000
마케팅    435.000000
영업     323.333333
인사     575.000000
Name: 월급, dtype: float64

--- 부서별 직원 수 ---
부서
IT     3
마케팅    2
영업     3
인사     2
Name: 이름, dtype: int64

--- 직급별 월급 최대/최소 ---
    max  min
직급          
과장  500  420
대리  450  300
부장  600  600
사원  280  250
차장  550  550


#### 6-2. 여러 집계 함수 한번에: `agg()`

- 여러 집계 함수를 동시에 적용하거나, 컬럼마다 다른 함수를 적용할 때 사용

In [56]:
# 1. 부서별로 월급과 판매실적의 여러 통계 한번에
print('--- 부서별 월급 통계 ---')
display(df_gb.groupby('부서')['월급'].agg(['mean', 'max', 'min', 'count']))

# 2. 컬럼마다 다른 함수 적용 (딕셔너리 형태)
print('\n--- 컬럼별 다른 집계 함수 ---')
result = df_gb.groupby('부서').agg(
    평균월급=('월급', 'mean'),
    최고월급=('월급', 'max'),
    인원수=('이름', 'count'),
    평균판매실적=('판매실적', 'mean')
).round(1)
display(result)

# 3. 복수 컬럼으로 groupby
print('\n--- 부서 + 직급별 평균 월급 ---')
display(df_gb.groupby(['부서', '직급'])['월급'].mean().reset_index())

--- 부서별 월급 통계 ---


,mean,max,min,count
부서,,,,
IT,410.000000,500,280,3
마케팅,435.000000,480,390,2
영업,323.333333,420,250,3
인사,575.000000,600,550,2



--- 컬럼별 다른 집계 함수 ---


,평균월급,최고월급,인원수,평균판매실적
부서,,,,
IT,410.0,500,3,760.0
마케팅,435.0,480,2,680.0
영업,323.3,420,3,916.7
인사,575.0,600,2,NaN



--- 부서 + 직급별 평균 월급 ---


,부서,직급,월급
0,IT,과장,500.0
1,IT,대리,450.0
2,IT,사원,280.0
3,마케팅,과장,480.0
4,마케팅,대리,390.0
5,영업,과장,420.0
6,영업,대리,300.0
7,영업,사원,250.0
8,인사,부장,600.0
9,인사,차장,550.0


#### 6-3. 그룹 통계값으로 새 열 추가: `transform()`

- `groupby().agg()`는 그룹별로 **하나의 집계값**을 반환 (행 수 줄어듦)
- `groupby().transform()`는 **원본과 같은 행 수**를 유지하면서 그룹별 통계값을 각 행에 붙임

<div class="alert alert-block alert-info">
<b>💡 transform 활용 사례</b>
"각 직원의 월급이 본인 부서 평균 대비 몇 %인가?" 같은 질문에 유용.
</div>

In [58]:
df_trans = df_gb.copy()

# 1. 부서별 평균 월급을 각 행에 추가 (transform)
df_trans['부서_평균월급'] = df_trans.groupby('부서')['월급'].transform('mean').round(0)

# 2. 개인 월급이 부서 평균 대비 몇 %인지 계산
df_trans['월급_비율(%)'] = (df_trans['월급'] / df_trans['부서_평균월급'] * 100).round(1)

# 3. 부서 평균보다 높으면 '우수', 낮으면 '보통'
df_trans['평가'] = np.where(df_trans['월급'] >= df_trans['부서_평균월급'], '우수', '보통')

print('=== transform 활용 결과 ===')
display(df_trans.sort_values('부서'))

# agg vs transform 비교
print('\n--- agg: 그룹별 하나의 값 (행 축소) ---')
print(df_trans.groupby('부서')['월급'].mean())

print('\n--- transform: 원본과 같은 행 수 유지 ---')
print(df_trans.groupby('부서')['월급'].transform('mean'))

=== transform 활용 결과 ===


,이름,부서,직급,월급,판매실적,부서_평균월급,월급_비율(%),평가
1,이영희,IT,과장,500,NaN,410.0,122.0,우수
4,안다솜,IT,대리,450,760.0,410.0,109.8,우수
7,오세진,IT,사원,280,NaN,410.0,68.3,보통
5,황준혁,마케팅,과장,480,640.0,435.0,110.3,우수
8,류하늘,마케팅,대리,390,720.0,435.0,89.7,보통
0,김철수,영업,대리,300,850.0,323.0,92.9,보통
2,박민수,영업,사원,250,920.0,323.0,77.4,보통
6,정미래,영업,과장,420,980.0,323.0,130.0,우수
3,최지우,인사,부장,600,NaN,575.0,104.3,우수
9,홍길동,인사,차장,550,NaN,575.0,95.7,보통



--- agg: 그룹별 하나의 값 (행 축소) ---
부서
IT     410.000000
마케팅    435.000000
영업     323.333333
인사     575.000000
Name: 월급, dtype: float64

--- transform: 원본과 같은 행 수 유지 ---
0    323.333333
1    410.000000
2    323.333333
3    575.000000
4    410.000000
5    435.000000
6    323.333333
7    410.000000
8    435.000000
9    575.000000
Name: 월급, dtype: float64


> #### Exercise
> 아래 `df_ex6` 데이터를 사용하여 다음을 수행하시오.
> 1. 카테고리별 총 매출액과 평균 매출액을 구하시오.
> 2. 지역별 판매 건수(count)와 최대 매출액을 구하시오.
> 3. 카테고리 + 지역별로 groupby 하여 평균 매출액을 구하시오.

```python
df_ex6 = pd.DataFrame({
    '상품': ['노트북A', '마우스B', '노트북C', '키보드D', '노트북E', '마우스F', '모니터G', '키보드H'],
    '카테고리': ['전자', '주변기기', '전자', '주변기기', '전자', '주변기기', '전자', '주변기기'],
    '지역': ['서울', '부산', '서울', '대구', '부산', '서울', '대구', '부산'],
    '매출': [1200000, 35000, 980000, 120000, 1100000, 42000, 250000, 115000]
})
```

In [60]:
df_ex6 = pd.DataFrame({
    '상품': ['노트북A', '마우스B', '노트북C', '키보드D', '노트북E', '마우스F', '모니터G', '키보드H'],
    '카테고리': ['전자', '주변기기', '전자', '주변기기', '전자', '주변기기', '전자', '주변기기'],
    '지역': ['서울', '부산', '서울', '대구', '부산', '서울', '대구', '부산'],
    '매출': [1200000, 35000, 980000, 120000, 1100000, 42000, 250000, 115000]
})
display(df_ex6)

# 여기에 코드를 작성하시오.


,상품,카테고리,지역,매출
0,노트북A,전자,서울,1200000
1,마우스B,주변기기,부산,35000
2,노트북C,전자,서울,980000
3,키보드D,주변기기,대구,120000
4,노트북E,전자,부산,1100000
5,마우스F,주변기기,서울,42000
6,모니터G,전자,대구,250000
7,키보드H,주변기기,부산,115000


> #### Advance
> - 아래 `df_adv6` 데이터에서 **각 지역별로 매출 상위 2개 상품**만 추출하시오.
> - 힌트: `groupby` 후 `apply`와 `nlargest`를 조합하거나, `rank()`를 활용해보시오.

```python
df_adv6 = pd.DataFrame({
    '상품': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I'],
    '지역': ['서울', '서울', '서울', '부산', '부산', '부산', '대구', '대구', '대구'],
    '매출': [500, 300, 800, 200, 700, 400, 600, 100, 900]
})
```

In [62]:
df_adv6 = pd.DataFrame({
    '상품': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I'],
    '지역': ['서울', '서울', '서울', '부산', '부산', '부산', '대구', '대구', '대구'],
    '매출': [500, 300, 800, 200, 700, 400, 600, 100, 900]
})
display(df_adv6)

# 여기에 코드를 작성하시오.


,상품,지역,매출
0,A,서울,500
1,B,서울,300
2,C,서울,800
3,D,부산,200
4,E,부산,700
5,F,부산,400
6,G,대구,600
7,H,대구,100
8,I,대구,900


---
## 7. 데이터 병합 (merge / concat)
- 실무에서 데이터는 보통 **여러 테이블에 분산**되어 있음.
- `pd.concat()`: 데이터프레임을 **위/아래 또는 좌/우**로 단순 연결
- `pd.merge()`: 공통 키(Key) 컬럼을 기준으로 **두 테이블을 조인** (SQL JOIN과 동일)

#### 7-1. 단순 연결: `pd.concat()`

```python
pd.concat([df1, df2], axis=0)  # 세로로 쌓기 (행 추가)
pd.concat([df1, df2], axis=1)  # 가로로 붙이기 (열 추가)
```

In [65]:
# 두 개의 분기별 데이터
df_q1 = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수'],
    '부서': ['영업', 'IT', '영업'],
    '1분기매출': [850, 720, 910]
})

df_q2 = pd.DataFrame({
    '이름': ['최지우', '안다솜', '황준혁'],
    '부서': ['인사', 'IT', '마케팅'],
    '1분기매출': [0, 760, 640]
})

# 1. 세로 연결 (axis=0, 기본값)
print('--- 세로 연결 (axis=0) ---')
df_concat_v = pd.concat([df_q1, df_q2], axis=0, ignore_index=True)
display(df_concat_v)

# 2. 가로 연결 (axis=1)
df_extra = pd.DataFrame({'2분기매출': [900, 800, 950]})
print('\n--- 가로 연결 (axis=1) ---')
df_concat_h = pd.concat([df_q1, df_extra], axis=1)
display(df_concat_h)

--- 세로 연결 (axis=0) ---


,이름,부서,1분기매출
0,김철수,영업,850
1,이영희,IT,720
2,박민수,영업,910
3,최지우,인사,0
4,안다솜,IT,760
5,황준혁,마케팅,640



--- 가로 연결 (axis=1) ---


,이름,부서,1분기매출,2분기매출
0,김철수,영업,850,900
1,이영희,IT,720,800
2,박민수,영업,910,950


#### 7-2. 키 기반 병합: `pd.merge()`

```python
pd.merge(left, right, on='키컬럼', how='inner')
```

| `how` 옵션 | 설명 | SQL 등가 |
| :--- | :--- | :--- |
| `'inner'` | 양쪽 모두에 있는 키만 유지 (기본값) | INNER JOIN |
| `'left'` | 왼쪽 테이블 기준, 없는 오른쪽 값은 NaN | LEFT JOIN |
| `'right'` | 오른쪽 테이블 기준, 없는 왼쪽 값은 NaN | RIGHT JOIN |
| `'outer'` | 양쪽 모두 포함, 없는 값은 NaN | FULL OUTER JOIN |

In [67]:
# 직원 기본 정보 테이블
df_emp = pd.DataFrame({
    '직원ID': [101, 102, 103, 104, 105],
    '이름':   ['김철수', '이영희', '박민수', '최지우', '안다솜'],
    '부서ID': [1, 2, 1, 3, 2]
})

# 부서 정보 테이블
df_dept = pd.DataFrame({
    '부서ID': [1, 2, 3, 4],
    '부서명': ['영업', 'IT', '인사', '마케팅'],
    '위치':  ['서울', '판교', '서울', '강남']
})

# 급여 정보 테이블 (모든 직원이 있지 않음)
df_salary = pd.DataFrame({
    '직원ID': [101, 102, 104, 106],
    '월급':   [300, 500, 400, 380],
    '성과급':  [50, 100, 80, 60]
})

print('직원 테이블:'); display(df_emp)
print('부서 테이블:'); display(df_dept)
print('급여 테이블:'); display(df_salary)

직원 테이블:


,직원ID,이름,부서ID
0,101,김철수,1
1,102,이영희,2
2,103,박민수,1
3,104,최지우,3
4,105,안다솜,2


부서 테이블:


,부서ID,부서명,위치
0,1,영업,서울
1,2,IT,판교
2,3,인사,서울
3,4,마케팅,강남


급여 테이블:


,직원ID,월급,성과급
0,101,300,50
1,102,500,100
2,104,400,80
3,106,380,60


In [68]:
# 1. inner join: 양쪽에 모두 있는 부서ID만 포함
print('--- INNER JOIN (직원 + 부서) ---')
df_inner = pd.merge(df_emp, df_dept, on='부서ID', how='inner')
display(df_inner)

# 2. left join: 직원 테이블 기준 (급여 없는 직원도 포함)
print('\n--- LEFT JOIN (직원 + 급여) ---')
df_left = pd.merge(df_emp, df_salary, on='직원ID', how='left')
display(df_left)

# 3. outer join: 양쪽 모두 포함
print('\n--- OUTER JOIN (직원 + 급여) ---')
df_outer = pd.merge(df_emp, df_salary, on='직원ID', how='outer')
display(df_outer)

# 4. 키 컬럼 이름이 다를 때: left_on, right_on
df_dept2 = df_dept.rename(columns={'부서ID': 'ID'})
print('\n--- 키 컬럼명이 다를 때: left_on / right_on ---')
df_diff_key = pd.merge(df_emp, df_dept2, left_on='부서ID', right_on='ID', how='inner')
display(df_diff_key)

--- INNER JOIN (직원 + 부서) ---


,직원ID,이름,부서ID,부서명,위치
0,101,김철수,1,영업,서울
1,102,이영희,2,IT,판교
2,103,박민수,1,영업,서울
3,104,최지우,3,인사,서울
4,105,안다솜,2,IT,판교



--- LEFT JOIN (직원 + 급여) ---


,직원ID,이름,부서ID,월급,성과급
0,101,김철수,1,300.0,50.0
1,102,이영희,2,500.0,100.0
2,103,박민수,1,NaN,NaN
3,104,최지우,3,400.0,80.0
4,105,안다솜,2,NaN,NaN



--- OUTER JOIN (직원 + 급여) ---


,직원ID,이름,부서ID,월급,성과급
0,101,김철수,1.0,300.0,50.0
1,102,이영희,2.0,500.0,100.0
2,103,박민수,1.0,NaN,NaN
3,104,최지우,3.0,400.0,80.0
4,105,안다솜,2.0,NaN,NaN
5,106,NaN,NaN,380.0,60.0



--- 키 컬럼명이 다를 때: left_on / right_on ---


,직원ID,이름,부서ID,ID,부서명,위치
0,101,김철수,1,1,영업,서울
1,102,이영희,2,2,IT,판교
2,103,박민수,1,1,영업,서울
3,104,최지우,3,3,인사,서울
4,105,안다솜,2,2,IT,판교


> #### Exercise
> 아래 세 테이블을 사용하여 다음을 수행하시오.
> 1. `df_order`와 `df_customer`를 고객ID로 LEFT JOIN 하시오.
> 2. 1번 결과에 `df_product`를 상품ID로 INNER JOIN 하시오.
> 3. 최종 결과에서 고객명, 지역, 상품명, 카테고리, 주문금액만 추출하시오.

```python
df_order = pd.DataFrame({
    '주문ID': ['O1', 'O2', 'O3', 'O4', 'O5'],
    '고객ID': ['C1', 'C2', 'C1', 'C3', 'C4'],
    '상품ID': ['P1', 'P2', 'P3', 'P1', 'P2'],
    '주문금액': [1200000, 35000, 250000, 1200000, 35000]
})
df_customer = pd.DataFrame({
    '고객ID': ['C1', 'C2', 'C3'],
    '고객명': ['김철수', '이영희', '박민수'],
    '지역': ['서울', '부산', '대구']
})
df_product = pd.DataFrame({
    '상품ID': ['P1', 'P2', 'P3'],
    '상품명': ['노트북', '마우스', '모니터'],
    '카테고리': ['전자', '주변기기', '전자']
})
```

In [70]:
df_order = pd.DataFrame({
    '주문ID': ['O1', 'O2', 'O3', 'O4', 'O5'],
    '고객ID': ['C1', 'C2', 'C1', 'C3', 'C4'],
    '상품ID': ['P1', 'P2', 'P3', 'P1', 'P2'],
    '주문금액': [1200000, 35000, 250000, 1200000, 35000]
})
df_customer = pd.DataFrame({
    '고객ID': ['C1', 'C2', 'C3'],
    '고객명': ['김철수', '이영희', '박민수'],
    '지역': ['서울', '부산', '대구']
})
df_product = pd.DataFrame({
    '상품ID': ['P1', 'P2', 'P3'],
    '상품명': ['노트북', '마우스', '모니터'],
    '카테고리': ['전자', '주변기기', '전자']
})

# 여기에 코드를 작성하시오.


> #### Advance
> - 위 Exercise에서 완성한 병합 결과를 활용하여 **지역별, 카테고리별 주문금액 합계**를 구하시오.
> - 단, 고객 정보가 없는 주문(C4)은 **제외**하고 집계하시오.

In [72]:
# 여기에 코드를 작성하시오.


---
## 8. apply / map 함수
- 조건이 복잡하거나 직접 만든 함수를 DataFrame에 적용할 때 사용함.
- `map()`: **Series**의 각 요소에 함수 또는 딕셔너리 변환 적용
- `apply()`: **Series** 또는 **DataFrame**에 함수 적용 (행/열 단위 연산 가능)

#### 8-1. 요소 단위 변환: `map()`

- Series의 각 요소에 함수, 딕셔너리, 또는 매핑을 적용
- 범주형 값을 다른 값으로 변환할 때 매우 편리

In [75]:
df_map = df_gb[['이름', '부서', '직급', '월급']].copy()

# 1. 딕셔너리로 매핑 (직급 → 직급코드)
rank_code = {'사원': 1, '대리': 2, '과장': 3, '차장': 4, '부장': 5}
df_map['직급코드'] = df_map['직급'].map(rank_code)

print('--- 딕셔너리 매핑 ---')
display(df_map)

# 2. 함수로 매핑 (월급 → 세금 계산)
def calc_tax(salary):
    if salary >= 500:
        return salary * 0.22  # 22% 세금
    elif salary >= 400:
        return salary * 0.18  # 18% 세금
    else:
        return salary * 0.15  # 15% 세금

df_map['세금'] = df_map['월급'].map(calc_tax)
df_map['실수령액'] = df_map['월급'] - df_map['세금']

print('\n--- 함수 매핑 (세금 계산) ---')
display(df_map)

--- 딕셔너리 매핑 ---


,이름,부서,직급,월급,직급코드
0,김철수,영업,대리,300,2
1,이영희,IT,과장,500,3
2,박민수,영업,사원,250,1
3,최지우,인사,부장,600,5
4,안다솜,IT,대리,450,2
5,황준혁,마케팅,과장,480,3
6,정미래,영업,과장,420,3
7,오세진,IT,사원,280,1
8,류하늘,마케팅,대리,390,2
9,홍길동,인사,차장,550,4



--- 함수 매핑 (세금 계산) ---


,이름,부서,직급,월급,직급코드,세금,실수령액
0,김철수,영업,대리,300,2,45.0,255.0
1,이영희,IT,과장,500,3,110.0,390.0
2,박민수,영업,사원,250,1,37.5,212.5
3,최지우,인사,부장,600,5,132.0,468.0
4,안다솜,IT,대리,450,2,81.0,369.0
5,황준혁,마케팅,과장,480,3,86.4,393.6
6,정미래,영업,과장,420,3,75.6,344.4
7,오세진,IT,사원,280,1,42.0,238.0
8,류하늘,마케팅,대리,390,2,58.5,331.5
9,홍길동,인사,차장,550,4,121.0,429.0


#### 8-2. 행/열 단위 함수 적용: `apply()`

```python
df['컬럼'].apply(함수)         # Series에 적용 (요소별)
df.apply(함수, axis=0)        # 열 방향으로 함수 적용
df.apply(함수, axis=1)        # 행 방향으로 함수 적용 (각 행에 함수 적용)
```

<div class="alert alert-block alert-info">
<b>💡 map vs apply</b>
- <b>map</b>: Series 전용, 요소 하나씩 처리 (단순 변환에 적합)
- <b>apply</b>: Series/DataFrame 모두 사용 가능, 행/열 전체를 처리 (복잡한 로직에 적합)
</div>

In [77]:
df_apply = df_gb.copy()

# 1. Series.apply(): 복잡한 로직의 등급 분류
def grade_salary(salary):
    if salary >= 500:
        return 'S등급'
    elif salary >= 400:
        return 'A등급'
    elif salary >= 300:
        return 'B등급'
    else:
        return 'C등급'

df_apply['월급등급'] = df_apply['월급'].apply(grade_salary)

# 2. lambda와 함께 (간단한 변환)
df_apply['월급_만원'] = df_apply['월급'].apply(lambda x: f'{x}만원')

print('--- Series.apply() 결과 ---')
display(df_apply[['이름', '월급', '월급등급', '월급_만원']])

# 3. DataFrame.apply(axis=1): 행 전체를 보는 복잡한 로직
def categorize_employee(row):
    # 부서가 IT이고 과장 이상이면 핵심인재
    if row['부서'] == 'IT' and row['직급'] in ['과장', '차장', '부장']:
        return '핵심인재'
    elif row['월급'] >= 450:
        return '고소득'
    else:
        return '일반'

df_apply['분류'] = df_apply.apply(categorize_employee, axis=1)

print('\n--- DataFrame.apply(axis=1) 결과 ---')
display(df_apply[['이름', '부서', '직급', '월급', '분류']])

--- Series.apply() 결과 ---


,이름,월급,월급등급,월급_만원
0,김철수,300,B등급,300만원
1,이영희,500,S등급,500만원
2,박민수,250,C등급,250만원
3,최지우,600,S등급,600만원
4,안다솜,450,A등급,450만원
5,황준혁,480,A등급,480만원
6,정미래,420,A등급,420만원
7,오세진,280,C등급,280만원
8,류하늘,390,B등급,390만원
9,홍길동,550,S등급,550만원



--- DataFrame.apply(axis=1) 결과 ---


,이름,부서,직급,월급,분류
0,김철수,영업,대리,300,일반
1,이영희,IT,과장,500,핵심인재
2,박민수,영업,사원,250,일반
3,최지우,인사,부장,600,고소득
4,안다솜,IT,대리,450,고소득
5,황준혁,마케팅,과장,480,고소득
6,정미래,영업,과장,420,일반
7,오세진,IT,사원,280,일반
8,류하늘,마케팅,대리,390,일반
9,홍길동,인사,차장,550,고소득


> #### Exercise
> 아래 `df_ex8` 데이터를 사용하여 다음을 수행하시오.
> 1. `점수` 컬럼에 `map()`을 사용하여 90점 이상 'A', 80점 이상 'B', 70점 이상 'C', 그 미만 'D' 등급을 부여하는 `등급` 컬럼을 추가하시오.
> 2. `apply(axis=1)`을 사용하여 `수학`과 `영어` 점수 중 **더 높은 값**을 담는 `최고점수` 컬럼을 추가하시오.
> 3. `map()`으로 `학과` 컬럼을 단과대학으로 변환하는 `단과대` 컬럼을 추가하시오. (컴공·전자 → 공과대학, 수학·통계 → 이과대학)

```python
df_ex8 = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수', '최지우', '안다솜'],
    '학과': ['컴공', '전자', '수학', '통계', '컴공'],
    '수학': [88, 92, 75, 95, 68],
    '영어': [75, 85, 90, 70, 82],
    '점수': [82, 89, 83, 83, 75]
})
```

In [79]:
df_ex8 = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수', '최지우', '안다솜'],
    '학과': ['컴공', '전자', '수학', '통계', '컴공'],
    '수학': [88, 92, 75, 95, 68],
    '영어': [75, 85, 90, 70, 82],
    '점수': [82, 89, 83, 83, 75]
})
display(df_ex8)

# 여기에 코드를 작성하시오.


,이름,학과,수학,영어,점수
0,김철수,컴공,88,75,82
1,이영희,전자,92,85,89
2,박민수,수학,75,90,83
3,최지우,통계,95,70,83
4,안다솜,컴공,68,82,75


> #### Advance
> - 아래 `df_adv8`의 `전화번호` 컬럼은 형식이 제각각임.
> - `apply()`와 함수를 활용하여 모든 전화번호를 **'010-XXXX-XXXX'** 형식으로 정규화하시오.
> - 힌트: 먼저 숫자만 추출한 후, 형식에 맞게 재조합하시오.

```python
df_adv8 = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수', '최지우', '안다솜'],
    '전화번호': ['010-1234-5678', '01056789012', '010.9012.3456', '010 3456 7890', '(010)7890-1234']
})
```

In [81]:
df_adv8 = pd.DataFrame({
    '이름': ['김철수', '이영희', '박민수', '최지우', '안다솜'],
    '전화번호': ['010-1234-5678', '01056789012', '010.9012.3456', '010 3456 7890', '(010)7890-1234']
})
display(df_adv8)

# 여기에 코드를 작성하시오.


,이름,전화번호
0,김철수,010-1234-5678
1,이영희,01056789012
2,박민수,010.9012.3456
3,최지우,010 3456 7890
4,안다솜,(010)7890-1234


---
## 종합 실습
### 온라인 쇼핑몰 주문 데이터 정제 프로젝트

아래는 온라인 쇼핑몰의 실제 주문 데이터.  
지금까지 배운 **결측치 처리, 중복 제거, 타입 변환, 문자열 처리, 날짜 처리, GroupBy, 병합, apply** 기법을 모두 활용하여 데이터를 정제하고 분석해볼 것.

In [83]:
# 종합 실습 데이터
order_data = {
    '주문ID':   ['O001', 'O002', 'O003', 'O004', 'O005', 'O006', 'O007', 'O008', 'O009', 'O010',
                 'O003', 'O011', 'O012', 'O013', 'O014', 'O015', 'O016', 'O017', 'O018', 'O019'],
    '고객명':   ['  김철수', '이영희', '박민수 ', None, '안다솜', '황준혁', '  정미래', '오세진',
                 '류하늘', '홍길동', '박민수 ', '신지아', '강민준', None, '최다인',
                 '윤서준', '박민수', '한소희', '임재원', '이영희'],
    '상품명':   ['노트북 Pro', '무선마우스', '27인치 모니터', '기계식 키보드', '노트북 Pro', '무선마우스',
                 '웹캠 HD', '27인치 모니터', '노트북 Pro', '기계식 키보드', '27인치 모니터',
                 '노트북 Pro', '무선마우스', '27인치 모니터', '노트북 Pro', '기계식 키보드',
                 '무선마우스', '웹캠 HD', '노트북 Pro', '무선마우스'],
    '카테고리': ['전자', '주변기기', '전자', '주변기기', '전자', '주변기기', '주변기기', '전자',
                 '전자', '주변기기', '전자', '전자', '주변기기', '전자', '전자',
                 '주변기기', '주변기기', '주변기기', '전자', '주변기기'],
    '가격':    ['1200000', '35000', '250000', '120000', '1200000', '35000', '85000', '250000',
                '1200000', '120000', '250000', '1200000', '35000', '250000', 'N/A',
                '120000', '35000', '85000', '1200000', '35000'],
    '수량':    [1, 2, 1, 3, 2, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 2, 3, 1, 2, 1],
    '주문일':  ['2024-01-05', '2024-01-08', '2024-01-12', '2024-01-15', '2024-02-03',
                '2024-02-10', '2024-02-20', '2024-03-01', '2024-03-15', '2024-03-22',
                '2024-01-12', '2024-04-05', '2024-04-12', '2024-04-25', '2024-05-02',
                '2024-05-18', '2024-06-01', '2024-06-15', '2024-06-20', '2024-06-28'],
    '지역':   ['서울', '부산', '서울', '대구', '부산', '서울', '대구', '서울', '부산', '대구',
               '서울', '서울', '부산', '대구', '서울', '부산', '서울', '대구', '부산', '서울'],
    '배송상태': ['배송완료', '배송중', '배송완료', '배송완료', '배송완료', '배송중', '배송완료',
                '배송완료', '준비중', '배송완료', '배송완료', '배송완료', '배송중', '배송완료',
                '준비중', '배송완료', '배송완료', '준비중', '배송완료', '배송중']
}
df_shop = pd.DataFrame(order_data)

print('=== 원본 주문 데이터 ===')
display(df_shop)
print(f'\n크기: {df_shop.shape}')
print(f'\n데이터 타입:\n{df_shop.dtypes}')
print(f'\n결측치 수:\n{df_shop.isnull().sum()}')

=== 원본 주문 데이터 ===


,주문ID,고객명,상품명,카테고리,가격,수량,주문일,지역,배송상태
0,O001,김철수,노트북 Pro,전자,1200000,1,2024-01-05,서울,배송완료
1,O002,이영희,무선마우스,주변기기,35000,2,2024-01-08,부산,배송중
2,O003,박민수,27인치 모니터,전자,250000,1,2024-01-12,서울,배송완료
3,O004,None,기계식 키보드,주변기기,120000,3,2024-01-15,대구,배송완료
4,O005,안다솜,노트북 Pro,전자,1200000,2,2024-02-03,부산,배송완료
5,O006,황준혁,무선마우스,주변기기,35000,1,2024-02-10,서울,배송중
6,O007,정미래,웹캠 HD,주변기기,85000,2,2024-02-20,대구,배송완료
7,O008,오세진,27인치 모니터,전자,250000,1,2024-03-01,서울,배송완료
8,O009,류하늘,노트북 Pro,전자,1200000,1,2024-03-15,부산,준비중
9,O010,홍길동,기계식 키보드,주변기기,120000,2,2024-03-22,대구,배송완료



크기: (20, 9)

데이터 타입:
주문ID    object
고객명     object
상품명     object
카테고리    object
가격      object
수량       int64
주문일     object
지역      object
배송상태    object
dtype: object

결측치 수:
주문ID    0
고객명     2
상품명     0
카테고리    0
가격      0
수량      0
주문일     0
지역      0
배송상태    0
dtype: int64


### 종합 실습 문제

위 `df_shop` 데이터를 단계별로 정제하고 분석하시오.

**[Step 1] 데이터 정제**
> 1. `주문ID` 기준으로 중복된 행을 제거하시오. (처음 등장한 행 유지)
> 2. `고객명` 컬럼의 앞뒤 공백을 제거하시오.
> 3. `고객명`이 없는(None) 행을 제거하시오.
> 4. `가격` 컬럼을 숫자형으로 변환하시오. (변환 불가 값은 NaN → 중앙값으로 채우기)
> 5. `주문일` 컬럼을 datetime 타입으로 변환하시오.

**[Step 2] 파생 변수 생성**
> 6. `가격 × 수량`으로 `주문금액` 컬럼을 추가하시오.
> 7. `주문일`에서 `주문_월`, `주문_분기`, `주문_요일` 컬럼을 추가하시오.
> 8. `주문금액` 기준으로 100만원 이상 '프리미엄', 10만원 이상 '일반', 그 미만 '소액' 으로 `주문등급` 컬럼을 추가하시오.

**[Step 3] 분석**
> 9. 카테고리별 총 주문금액과 평균 주문금액을 구하시오.
> 10. 지역별 주문 건수와 총 주문금액을 구하시오.
> 11. 월별 주문금액 합계를 계산하여 가장 매출이 높은 달을 출력하시오.
> 12. 배송 상태별 주문 건수를 출력하고, '배송완료' 비율(%)을 계산하시오.

In [85]:
# [Step 1] 데이터 정제
df_clean = df_shop.copy()

# 1. 중복 제거

# 2. 고객명 공백 제거

# 3. 고객명 None 행 제거

# 4. 가격 숫자형 변환 (NaN → 중앙값)

# 5. 주문일 datetime 변환


# [Step 2] 파생 변수 생성
# 6. 주문금액

# 7. 주문 날짜 요소

# 8. 주문등급


# [Step 3] 분석
# 9. 카테고리별 통계

# 10. 지역별 통계

# 11. 월별 매출

# 12. 배송 상태 분석
